# 1. Research Question
Does the association between a trader's exposure to a topic and **overall trader profitability** differ across topics?

`trader_ppv` is the primary outcome; `trader_pnl` is secondary. Both describe overall trader performance. This notebook does **not** estimate profit from trades within a particular topic. That would require linked transaction-level or topic-level profit data.

These are exploratory, unadjusted associations among traders with at least 100 transactions, not causal effects of specialization. A larger exposure value represents greater measured exposure to that topic, subject to the definition limitation below.

# 2. Data Loading
Run from the directory containing `data_no_imputation.parquet`. Dependencies: pandas, NumPy, Matplotlib, SciPy, a pandas Parquet engine (such as pyarrow), and Jupyter/IPython. All analysis state is created here; the source file is only read.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from scipy import stats
from itertools import combinations
from IPython.display import display, Markdown

plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False})
df = pd.read_parquet("data_no_imputation.parquet")
print('Dataset shape:', df.shape)
display(pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum()}))
topic_columns = [c for c in df.columns if c.startswith('topic_')]
required = ['transaction_count', 'trader_ppv', 'trader_pnl'] + topic_columns
if not topic_columns:
    raise ValueError('No topic_ columns found.')
if not set(required).issubset(df.columns):
    raise ValueError('Required profitability or transaction columns are missing.')
print(f'Available topics ({len(topic_columns)}):', topic_columns)
def topic_name(column):
    return column.removeprefix('topic_').replace('_', ' ').title()

# 3. Data Preparation
Use one common complete-case sample across all topics and both outcomes so comparisons use the same traders. This can exclude a trader from the primary analysis because their secondary outcome is missing. Only relevant numeric columns determine missing/infinite exclusions; unrelated missing values do not matter. No finite extremes are removed, clipped, winsorized, or imputed. Each trader receives equal weight.

In [ ]:
numeric = df[required].apply(pd.to_numeric, errors='raise')
eligible = numeric['transaction_count'].ge(100)
finite = np.isfinite(numeric.to_numpy(dtype=float, na_value=np.nan)).all(axis=1)
analysis = numeric.loc[eligible & finite].copy()
print(f'Traders meeting transaction threshold: {eligible.sum():,}')
print(f'Removed for missing/infinite relevant values: {(eligible & ~finite).sum():,}')
print(f'Traders remaining after filtering: {len(analysis):,}')
if len(analysis) < 3:
    raise ValueError('Too few traders remain.')
if 'trader' in df.columns:
    ids = df.loc[analysis.index, 'trader']
    if ids.isna().any() or ids.duplicated().any():
        raise ValueError('Expected one row per identified trader; resolve missing/duplicate IDs before inference.')
else:
    print('No trader identifier: one independent trader per row is an unverified assumption.')
display(analysis[['transaction_count', 'trader_ppv', 'trader_pnl']].describe())

# 4. Topic Variable Interpretation
The available project notebooks document cleaning and use of specialization variables, but do not establish the denominator or aggregation rule for these `topic_` columns. Thus they cannot establish whether these are transaction shares, volume shares, or another activity measure. We call them **topic-exposure values** throughout.

Bounds and row sums are diagnostics, not proof of meaning. Values summing to one could also be normalized classification scores. We do not normalize them: overlapping topic assignments could legitimately produce sums above one. A 0.1-unit change must not be described as ten percentage points of transactions or volume without a verified definition.

In [ ]:
exposure = analysis[topic_columns]
distribution = exposure.describe(percentiles=[.01, .25, .5, .75, .99]).T
distribution['zero_fraction'] = exposure.eq(0).mean()
distribution['one_fraction'] = exposure.eq(1).mean()
display(distribution)
row_sums = exposure.sum(axis=1)
print('Sum of topic exposures per trader:')
display(row_sums.describe())
print(f'Fraction with sums approximately 1: {np.isclose(row_sums, 1).mean():.2%}')
unit_bounded = bool(((exposure >= 0) & (exposure <= 1)).all().all())
print('All retained topic values in [0, 1]:', unit_bounded)
print('Definitions remain unverified; no normalization performed.')

# 5. Visualizations
All figures use the full filtered sample and finite outcome range. Ten common equal-width exposure intervals are used across topics. The first interval includes both endpoints; subsequent intervals exclude the left endpoint and include the right. If values are outside [0, 1], the common interval range expands to include them. Zero exposure is included in the first bin and may dominate it.

Empty profitability cells stay missing, not zero. Counts reveal sparse groups; small-group means and medians may be unstable. Binned lines summarize intervals, not fitted relationships.

In [ ]:
lower = min(0.0, float(exposure.min().min()))
upper = max(1.0, float(exposure.max().max()))
edges = np.linspace(lower, upper, 11)
labels = [f"{'[' if i == 0 else '('}{edges[i]:.3g}, {edges[i+1]:.3g}]" for i in range(10)]
bin_summaries = {}
for topic in topic_columns:
    bins = pd.cut(analysis[topic], edges, labels=labels, include_lowest=True, right=True)
    assert bins.notna().all(), 'Binning must preserve every retained trader.'
    summary = analysis.groupby(bins, observed=False).agg(
        n=('trader_ppv', 'size'), mean_ppv=('trader_ppv', 'mean'),
        median_ppv=('trader_ppv', 'median'), median_pnl=('trader_pnl', 'median'))
    bin_summaries[topic] = summary.reindex(labels)
    assert summary['n'].sum() == len(analysis)

### Chart 1: Topic Participation vs. Profitability
Participation is measured by the available topic-exposure value, whose activity denominator is unknown. Each point is a trader's overall PPV.

In [ ]:
for topic in topic_columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(analysis[topic], analysis['trader_ppv'], s=9, alpha=.12, edgecolors='none', rasterized=True)
    ax.axhline(0, color='gray', lw=.8)
    ax.set(title=f'{topic_name(topic)}: exposure vs. overall profitability',
           xlabel='Topic-exposure value (definition unverified)', ylabel='Overall trader_ppv')
    fig.tight_layout()
    plt.show()
    plt.close(fig)

### Chart 2: Mean Profitability by Topic Participation

In [ ]:
for topic, summary in bin_summaries.items():
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(np.arange(10), summary['mean_ppv'], marker='o')
    ax.set_xticks(np.arange(10), [f'{label}\nn={count:,}' for label, count in zip(labels, summary['n'])], rotation=45, ha='right')
    ax.axhline(0, color='gray', lw=.8)
    ax.set(title=f'Mean overall profitability: {topic_name(topic)}',
           xlabel='Topic-exposure interval; n = traders', ylabel='Mean overall trader_ppv')
    fig.tight_layout()
    plt.show()
    plt.close(fig)

### Chart 3: Median Profitability by Topic Participation

In [ ]:
for topic, summary in bin_summaries.items():
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(np.arange(10), summary['median_ppv'], marker='o')
    ax.set_xticks(np.arange(10), [f'{label}\nn={count:,}' for label, count in zip(labels, summary['n'])], rotation=45, ha='right')
    ax.axhline(0, color='gray', lw=.8)
    ax.set(title=f'Median overall profitability: {topic_name(topic)}',
           xlabel='Topic-exposure interval; n = traders', ylabel='Median overall trader_ppv')
    fig.tight_layout()
    plt.show()
    plt.close(fig)

### Chart 4: Topic Participation Heatmaps

In [ ]:
median_matrix = pd.DataFrame({t: s['median_ppv'] for t, s in bin_summaries.items()}).T
count_matrix = pd.DataFrame({t: s['n'] for t, s in bin_summaries.items()}).T
for matrix, title, count_plot in [
    (median_matrix, 'Median overall trader_ppv by topic-exposure interval', False),
    (count_matrix, 'Number of traders by topic-exposure interval', True)]:
    fig, ax = plt.subplots(figsize=(17, max(6, .48 * len(topic_columns))))
    values = matrix.to_numpy(dtype=float)
    cmap = plt.get_cmap('viridis' if count_plot else 'RdBu_r').copy()
    cmap.set_bad('#dddddd')
    options = {}
    if not count_plot:
        limit = max(float(np.nanmax(np.abs(values))), 1e-12)
        options['norm'] = TwoSlopeNorm(vmin=-limit, vcenter=0, vmax=limit)
    im = ax.imshow(np.ma.masked_invalid(values), aspect='auto', cmap=cmap, **options)
    ax.set_xticks(range(len(labels)), labels, rotation=45, ha='right')
    ax.set_yticks(range(len(topic_columns)), [topic_name(t) for t in topic_columns])
    ax.set(title=title, xlabel='Topic-exposure interval', ylabel='Topic')
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            v = values[i, j]
            text = 'NA' if np.isnan(v) else (f'{v:,.0f}' if count_plot else f'{v:.3g}')
            rgba = im.cmap(im.norm(v)) if np.isfinite(v) else (1, 1, 1, 1)
            color = 'black' if np.dot(rgba[:3], [.299, .587, .114]) > .55 else 'white'
            ax.text(j, i, text, ha='center', va='center', fontsize=8, color=color)
    fig.colorbar(im, ax=ax, label='Number of traders' if count_plot else 'Median overall trader_ppv')
    fig.tight_layout()
    plt.show()
    plt.close(fig)

### Chart 5: Profitability by Primary Topic ? Conditional Proxy
An economic **primary activity topic** cannot be verified without comparable activity-share definitions. As a descriptive proxy, when all values lie in [0, 1], we group traders by their **largest recorded topic-exposure value**. This is conditional on cross-topic comparability, not confirmation of a primary transaction/volume topic. Ties (including numerical near-ties within absolute tolerance 1e-12) form a separate group; all-zero rows form a no-exposure group. No arbitrary tie-breaking is used.

In [ ]:
primary_summary = None
if unit_bounded:
    values = exposure.to_numpy()
    maxima = values.max(axis=1)
    tied = np.isclose(values, maxima[:, None], rtol=0, atol=1e-12).sum(axis=1) > 1
    groups = exposure.idxmax(axis=1).map(topic_name)
    groups.loc[tied] = 'Tied largest exposure'
    groups.loc[maxima == 0] = 'No recorded exposure'
    primary_summary = analysis.assign(primary_exposure=groups).groupby('primary_exposure').agg(
        median_ppv=('trader_ppv', 'median'), median_pnl=('trader_pnl', 'median'), n=('trader_ppv', 'size'))
    primary_summary = primary_summary.sort_values('median_ppv')
    display(primary_summary)
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.barh(range(len(primary_summary)), primary_summary['median_ppv'])
    ax.set_yticks(range(len(primary_summary)), [f'{g} (n={n:,})' for g, n in zip(primary_summary.index, primary_summary['n'])])
    ax.axvline(0, color='gray', lw=.8)
    ax.set(title='Overall profitability by largest recorded exposure (conditional primary-topic proxy)',
           xlabel='Median overall trader_ppv', ylabel='Largest-exposure group')
    fig.tight_layout()
    plt.show()
    plt.close(fig)
else:
    print('Primary-topic proxy skipped: comparable share-like bounds are not supported.')

# 6. Statistical Analysis
### Correlations and multiple testing
Pearson measures linear association; Spearman measures monotonic rank association. Constant inputs yield undefined results. Benjamini?Hochberg (BH) correction is applied jointly to all valid topic ? method ? outcome correlation tests, including secondary PNL tests. We also show the more conservative Benjamini?Yekutieli (BY) correction because topic tests are dependent and BH's dependence conditions are not guaranteed. P-values assume independent traders; shared markets or related wallets can violate that assumption. Effect sizes matter even with small p-values.

In [ ]:
def adjust_p(p_values, dependence=False):
    p = np.asarray(p_values, dtype=float)
    q = np.full(p.shape, np.nan)
    valid = np.flatnonzero(np.isfinite(p))
    order = valid[np.argsort(p[valid])]
    m = len(order)
    if m:
        factor = np.sum(1 / np.arange(1, m+1)) if dependence else 1.0
        raw = p[order] * m * factor / np.arange(1, m+1)
        q[order] = np.minimum(1, np.minimum.accumulate(raw[::-1])[::-1])
    return q

rows = []
for outcome in ['trader_ppv', 'trader_pnl']:
    for topic in topic_columns:
        x, y = analysis[topic], analysis[outcome]
        row = {'outcome': outcome, 'topic': topic, 'n_traders': len(analysis)}
        for method, function in [('pearson', stats.pearsonr), ('spearman', stats.spearmanr)]:
            r, p = function(x, y) if x.nunique() > 1 and y.nunique() > 1 else (np.nan, np.nan)
            row[method + '_r'], row[method + '_p'] = r, p
        rows.append(row)
correlations = pd.DataFrame(rows)
p_matrix = correlations[['pearson_p', 'spearman_p']].to_numpy()
for suffix, dependence in [('q_bh', False), ('q_by', True)]:
    adjusted = adjust_p(p_matrix.ravel(), dependence).reshape(p_matrix.shape)
    correlations[['pearson_' + suffix, 'spearman_' + suffix]] = adjusted
for outcome in ['trader_ppv', 'trader_pnl']:
    print('Overall outcome:', outcome)
    display(correlations.loc[correlations.outcome.eq(outcome)].reset_index(drop=True))

### Direct test of differences across topics
A significant association for one topic and a nonsignificant association for another does not establish a difference. We directly compare **marginal linear slopes** from separate intercept-plus-exposure regressions using the same traders. Slopes are reported per 0.1 exposure unit; this comparison assumes exposure scales are comparable.

The covariance is calculated jointly from trader-level OLS influence contributions, preserving dependence between topic estimates and allowing heteroskedastic residuals (HC0 sandwich). An omnibus Wald test asks whether all estimable topic slopes are equal. Pairwise slope-difference tests localize differences, with BH and BY corrections across all pairs and both outcomes; the two omnibus tests receive Bonferroni correction. These are large-sample tests and can be sensitive to extreme observations and rare topics. They test linear associations, not equality of nonlinear curves or causal effects. Correlations and binned medians provide complementary summaries.

This avoids treating topic?trader rows as independent observations. It does not control for trading volume, experience, market mix, or other confounders. If exposures sum to one, increasing one necessarily changes others; marginal slopes are not independent interventions.

In [ ]:
slope_tables, pair_rows, omnibus_rows = {}, [], []
for outcome in ['trader_ppv', 'trader_pnl']:
    X = exposure.to_numpy(dtype=float)
    xc = X - X.mean(axis=0)
    variance = np.mean(xc ** 2, axis=0)
    valid = variance > 0
    topics = np.array(topic_columns)[valid]
    xc, variance = xc[:, valid], variance[valid]
    y = analysis[outcome].to_numpy(dtype=float)
    yc = y - y.mean()
    slopes = np.mean(xc * yc[:, None], axis=0) / variance
    residuals = yc[:, None] - xc * slopes
    influence = xc * residuals / variance
    covariance = influence.T @ influence / len(y) ** 2
    se = np.sqrt(np.maximum(np.diag(covariance), 0))
    slope_tables[outcome] = pd.DataFrame({'topic': topics, 'slope_per_0.1': .1 * slopes,
        'ci95_low': .1 * (slopes - 1.96 * se), 'ci95_high': .1 * (slopes + 1.96 * se)})
    if len(topics) >= 2:
        C = np.eye(len(topics))[1:] - np.eye(len(topics))[0]
        delta, V = C @ slopes, C @ covariance @ C.T
        rank = np.linalg.matrix_rank(V)
        # Require full contrast rank: do not claim an all-slopes test from a singular covariance.
        wald = float(delta @ np.linalg.solve(V, delta)) if rank == len(C) else np.nan
        p = stats.chi2.sf(wald, rank) if np.isfinite(wald) else np.nan
        omnibus_rows.append({'outcome': outcome, 'n_topics': len(topics), 'wald': wald,
                              'df': rank, 'p': p, 'full_rank': rank == len(C)})
        for i, j in combinations(range(len(topics)), 2):
            difference = slopes[i] - slopes[j]
            difference_se = np.sqrt(max(covariance[i,i] + covariance[j,j] - 2*covariance[i,j], 0))
            p = 2 * stats.norm.sf(abs(difference / difference_se)) if difference_se > 0 else np.nan
            pair_rows.append({'outcome': outcome, 'topic_a': topics[i], 'topic_b': topics[j],
                'difference_per_0.1': .1 * difference, 'ci95_low': .1*(difference-1.96*difference_se),
                'ci95_high': .1*(difference+1.96*difference_se), 'p': p})
    print(f'{outcome}: marginal slopes with pointwise (not simultaneous) 95% intervals')
    display(slope_tables[outcome])
omnibus = pd.DataFrame(omnibus_rows)
pairwise = pd.DataFrame(pair_rows)
if not omnibus.empty:
    omnibus['p_bonferroni'] = (omnibus['p'] * len(omnibus)).clip(upper=1)
    display(omnibus)
if not pairwise.empty:
    pairwise['q_bh'] = adjust_p(pairwise['p'])
    pairwise['q_by'] = adjust_p(pairwise['p'], dependence=True)
    print('All pairwise differences (A minus B); intervals are pointwise, not multiplicity-adjusted:')
    with pd.option_context('display.max_rows', None):
        display(pairwise.sort_values(['outcome', 'q_by']))

# 7. Key Findings
The following summary is generated from the executed analysis, rather than assuming the direction or significance of results.

In [ ]:
findings = [f'**Sample:** {len(analysis):,} traders with at least 100 transactions and finite values in all relevant columns; {len(topic_columns)} topics.']
for outcome in ['trader_ppv', 'trader_pnl']:
    result = correlations.loc[correlations.outcome.eq(outcome)]
    valid = result.dropna(subset=['spearman_r'])
    if not valid.empty:
        strongest = valid.loc[valid.spearman_r.abs().idxmax()]
        findings.append(f"**{outcome}:** largest absolute Spearman association is {topic_name(strongest['topic'])}: rho={strongest['spearman_r']:.3f}, BH q={strongest['spearman_q_bh']:.3g}, BY q={strongest['spearman_q_by']:.3g}. Across topics, {int((result.spearman_q_by < .05).sum())} Spearman associations meet BY q < 0.05.")
    if not omnibus.empty:
        test = omnibus.loc[omnibus.outcome.eq(outcome)]
        if not test.empty:
            p = test.iloc[0]['p_bonferroni']
            verdict = 'evidence of differing marginal linear slopes' if p < .05 else 'insufficient evidence of differing marginal linear slopes'
            if np.isfinite(p):
                findings.append(f'**Across-topic test ({outcome}):** {verdict} (Bonferroni-adjusted omnibus p={p:.3g}).')
            else:
                findings.append(f'**Across-topic test ({outcome}):** covariance is singular; no full omnibus conclusion is available.')
    if not pairwise.empty:
        pairs = pairwise.loc[pairwise.outcome.eq(outcome)]
        findings.append(f"**Pairwise differences ({outcome}):** {int((pairs.q_by < .05).sum())} of {len(pairs)} pairs meet BY q < 0.05; inspect effect sizes and pointwise intervals above.")
findings += [
    '**Interpretation:** these are associations between recorded topic exposure and overall trader performance, not returns earned within a topic. The activity denominator remains unknown; even row sums of one cannot establish transaction or volume shares.',
    '**Limits:** marginal linear tests may miss nonlinear patterns. Sparse bins, finite extremes, selection into the 100-transaction sample, compositional exposure, confounding, and dependence across traders limit conclusions. Nonsignificance does not prove equality. PNL may reflect trading scale as well as profitability.',
    '**Next data needed:** a verified topic-exposure definition and trade-level realized profits linked to topic labels would allow a separate investigation of topic-specific returns.'
]
display(Markdown('\n\n'.join(findings)))